# 事前準備：データのダウンロード
以下を実行して、外部ファイルをダウンロードしてください。
**このセルはColaboratoryを起動するたびに必要となります**



In [1]:
import urllib.request
import os
import sys

os.makedirs('text', exist_ok=True)
urllib.request.urlretrieve('https://park.itc.u-tokyo.ac.jp/yamakata-lab/lecture/mediaproc/mediaproc3/gakumonno_susume_org.txt', './text/gakumonno_susume_org.txt')
urllib.request.urlretrieve('https://park.itc.u-tokyo.ac.jp/yamakata-lab/lecture/mediaproc/mediaproc3/genjimonogatari_org.txt', './text/genjimonogatari_org.txt')
urllib.request.urlretrieve('https://park.itc.u-tokyo.ac.jp/yamakata-lab/lecture/mediaproc/mediaproc3/ningen_shikkaku_org.txt', './text/ningen_shikkaku_org.txt')
urllib.request.urlretrieve('https://park.itc.u-tokyo.ac.jp/yamakata-lab/lecture/mediaproc/mediaproc3/wagahaiwa_nekodearu_org.txt', './text/wagahaiwa_nekodearu_org.txt')

##################################
### Colaboratoryのみ以下を実行 ###
##################################
if 'google.colab' in sys.modules:
    !pip install janome

# 第3回課題：与謝野晶子訳 「源氏物語 桐壺」っぽい文を生成してみよう！

**課題は結果が出力されている状態で保存して提出してください！**

## 課題１：分かち書き文の生成

`text/genjimonogatari_org.txt`には 与謝野晶子訳 著「源氏物語 桐壺」の本文が記録されています。
このテキストを半角スペース区切りの分かち書き文に変換して、`text/genjimonogatari_wakati.txt`に保存してください。
なお、改行は削除せずにそのまま残してしておいてください。

In [2]:
# -*- coding:utf-8 -*-
# このセルは書き換えないでください
# janomeのTokenizer()は1回だけ実行して、あとはオブジェクトtを使いまわします
# よって、このセルは1度のみ実行します

from janome.tokenizer import Tokenizer

t = Tokenizer()

infile = './text/genjimonogatari_org.txt'
outfile = './text/genjimonogatari_wakati.txt'


In [8]:
# ここに解答を書いてください。
with open(infile, 'r', encoding='utf-8') as f_in, open(outfile, 'w', encoding='utf-8') as f_out:
    for line in f_in:
        words = t.tokenize(line, wakati=True)
        wakati_line = ' '.join(words)
        f_out.write(wakati_line)

以下のコードで冒頭の30文字を出力してください。
次のような単語列が出てきたら正解です。  
```どの 天皇 様 の 御代 で あっ た か 、 女御 とか ```

In [9]:
with open(outfile, 'r', encoding='utf-8') as fi:
    print(fi.read(30))

どの 天皇 様 の 御代 で あっ た か 、 女御 とか 


## 課題２：ランダム文生成

下のセルで、課題１で作成した`text/genjimonogatari_wakati.txt`を入力として、統計的tri-gramモデルを学習してください。  
学習モデルの変数名は`lm`とします。  
なお、各行の先頭には`<BOP> `、末尾には`<EOP> `を追加してください。

In [10]:
# ここにtri-gramモデルを学習するコードを書いてください。
from nltk.lm import Vocabulary
from nltk.lm.models import MLE
from nltk.util import ngrams


words = [('<BOP> ' + l + ' <EOP>').split() for l in open(outfile, 'r', encoding='utf-8').readlines()]
vocab = Vocabulary([item for sublist in words for item in sublist])
text_trigrams = [ngrams(word, 3) for word in words]

n = 3
lm = MLE(order = n, vocabulary = vocab) # 最尤推定法（Maximum Likelihood Estimation)による統計的n-gram言語モデルの準備
lm.fit(text_trigrams) # 上で生成したtri-gramを使って言語モデルを学習


学習したモデル`lm`を使って、以下のセルで「帝は」から始まる文を10文生成してください。
- 生成文は「帝は」から始めて、『`。`』か『 `」`』か『 `<EOS>`』のいずれかで終わる一文を基本としますが、より自然な文となるよう、独自の改良を加えても構いません。
- 生成文は単語のリストでもなく、分かち書き文（単語の区切りに半角スペースが挿入された文）でもなく、通常の文の形にしてください（下の例の通りです）

**生成文が出力されている状態で保存して提出してください。**

In [11]:
# ここに上のセルで学習したモデルを使ってランダム文を生成するコードを書いてください。
for _ in range(10):
    context = ['帝', 'は']
    sentence = context.copy()
    while True:
        w = lm.generate(text_seed=context)
        sentence.append(w)
        context = sentence[-2:]
        if w in ['。', '」', '<EOP>']:
            break
    print(''.join(sentence) + '\n')


帝はまたどんな呪詛が行なわれる、そのお光でみすぼらしさも隠していただいて、実家へ下がって拝礼をしていた。

帝はお弔いの品々が下された酒宴に、平生よりも美しいお顔を拝見する機会を多く得ていた人でも、自然にわくであろう。「死んだから、帝にまた楽しい御生活にお申し入れになっていた。

帝は思召して美しい評判のある人は尊敬をささげている心細い更衣はお返辞を躊躇して帝がおとりなしになる方の次の席がそのお前にできていたが、悲しみの深くなるのであるからと見てはいっそう憐れを多くお加えになった。

帝は一の皇子のほうは修理の役所、内匠寮などへ帝がしばしばそこへおいでになるものはなかった。「死んだのちの彼を世話する人として見て、「限りとて別るる道の悲しきにいかまほしきは命なりけり死がそれほど私に長生きさせるのが重荷になりまして、またそれかといって、心の、悲しい暗さがせめて一部分でもいらっしゃいますでしょう。

帝は思召してみてからは失敬な女ではないだろうかという御様子を思っていられた式場へ着いた時から親たちにひけをとらせないよき保護者たりえた。

帝はお引き留めになった。

帝はお許しにならず、大事にされたなどと蔭ではいわれる。

帝は命婦にこまごまと大納言家の人は男女の別なしに何かの妻妾から生まれた子供を幾人も人間世界に生まれた令嬢があったが、こんなことでございますのに、こうして一人で更衣の死を悲しむのはよい一対のうるわしいことであるが、今日から始めるはずの祈祷も高僧たちが承っていたから、私を置いておきたく思召した。

帝はこの人を常のこともやはり太子には今蔵人少将であったらこのまま自分の思ったことがございませんようなことになって桐壺の更衣のもとの桐壺の宮へ差し上げました母親のないというふうをあそばして、幸福な道を選ぼうとして命婦は泣く泣く、「もうしばらく御所で養生をしていたが、皇子の席の折り詰めのおそばをお聞きになっておられぬ二人の常の御殿の住人たちの内親王は当帝の御入内のことがございませんでしたのに、こうして一人でも生きていることであるから、将来に最も頼もしい位置をこの子を置きたくないような死に方を、亭子院のおそばに奉仕しておくことがあった。

帝はこの世界には故更衣のもとの桐壺の更衣が出かけて行く桐壺であるが、せめてその子のお給仕がつくのである。



たとえば次のような文が生成されます（これは出力例です）。
```
帝はそれほどお驚きにならないと、帝のお言づてのほかの御消息を渡した。
帝は思召して話はそのままになって政務も何度も何も出して言うことのできなかった。
帝は思召したが、自然どの女御の宮と一方を申していた。
帝は桐壺の更衣の退出ができなかった。
帝はある程度まではおさえておいでになりたい思召しておいでになった。
帝は想像あそばしながら起きておいでになった更衣はこう呼ばれるのであった。
帝はお言い及ぼしになって、さてそれでよいかと皆驚いていたから、それでは十分でないほどのものにされて、こんな悲しい勅使であなたをお迎えすると、帝のおとりなしも不思議なほどあなたとこの子と見ても愚か者はただ遺言を苦労して帝がおとりなしに何かののちに第二の皇子の御朝餐として用意される皇子の将来を見通して、故大納言家に行っているようなことをお許しにならなかったからである。
帝はお読みになって、御覚醒になる方であるから、年下の少年に配されたのだ、自分の心になる時、弘徽殿の女御からお使いが荒ら屋へおいでなさい。
帝は想像あそばしながら起きておいでになって、非常なりっぱなものであった。
帝はまして御自制なされがたい御感情があった。
```

## おまけ
以下の原文も用意しています。「源氏物語 桐壺」とどの程度違うか、ぜひ試してみてください。
**<font color='red'>おまけのコードは課題提出の際は削除してください。</font>**
- `text/gakumonno_susume_org.txt`：福沢諭吉 著「学問のすすめ」
- `text/wagahaiwa_nekodearu_org.txt`：夏目漱石 著「吾輩は猫である」
- `text/ningen_shikkaku_org.txt`: 太宰治 著「人間失格」

<!--2026S2-->